# 17 — Pivot Tables, Crosstab & Melt

Wide ↔ Long reshaping is a core data engineering skill. This notebook covers
pivot_table, crosstab, melt, wide_to_long, and stack/unstack.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 600

sales = pd.DataFrame({
    'order_id':   range(5001, 5001 + n),
    'salesperson': np.random.choice(['Alice', 'Bob', 'Carol', 'Dave', 'Eve'], n),
    'region':     np.random.choice(['North', 'South', 'East', 'West'], n),
    'category':   np.random.choice(['Electronics', 'Clothing', 'Food', 'Books'], n),
    'revenue':    np.round(np.random.uniform(500, 80000, n), 2),
    'units':      np.random.randint(1, 50, n),
    'quarter':    np.random.choice(['Q1', 'Q2', 'Q3', 'Q4'], n),
    'year':       np.random.choice([2022, 2023], n),
})

print(f"Shape: {sales.shape}")
sales.head(3)

## 1. pivot_table() — Summary Aggregation in 2D

In [ ]:
# Revenue by region (rows) × category (columns)
pt = pd.pivot_table(
    sales,
    values='revenue',
    index='region',
    columns='category',
    aggfunc='sum',
    margins=True,      # Row/column totals
    margins_name='Total'
).round(0)

print(pt)

In [ ]:
# Multiple values — revenue + units
pt2 = pd.pivot_table(
    sales,
    values=['revenue', 'units'],
    index='region',
    columns='category',
    aggfunc={'revenue': 'sum', 'units': 'mean'},
    fill_value=0
).round(1)
print(pt2)

In [ ]:
# Multiple index levels — salesperson × region
pt3 = pd.pivot_table(
    sales,
    values='revenue',
    index=['year', 'quarter'],
    columns='region',
    aggfunc='sum',
    fill_value=0
).round(0)
print(pt3)

In [ ]:
# Multiple aggfunc per value
pt4 = pd.pivot_table(
    sales,
    values='revenue',
    index='salesperson',
    columns='category',
    aggfunc=['mean', 'count'],
    fill_value=0
).round(0)
print(pt4)

## 2. df.pivot() — Reshape Without Aggregation

In [ ]:
# pivot() requires unique (index, columns) combinations
# pivot_table() handles duplicates (aggregates them)

# Create a dataset with exactly one value per (salesperson, category)
summary = (
    sales.groupby(['salesperson', 'category'])['revenue']
    .sum().reset_index()
)

wide = summary.pivot(
    index='salesperson',
    columns='category',
    values='revenue'
).round(0)
print(wide)

## 3. crosstab() — Frequency Tables

In [ ]:
# Basic crosstab — counts
ct = pd.crosstab(sales['region'], sales['category'])
print(ct)

In [ ]:
# Normalized — percentage within each row
ct_norm = pd.crosstab(
    sales['region'],
    sales['category'],
    normalize='index'  # 'index', 'columns', 'all'
).round(3) * 100
print(ct_norm)

In [ ]:
# Crosstab with values (like pivot_table)
ct_val = pd.crosstab(
    sales['region'],
    sales['category'],
    values=sales['revenue'],
    aggfunc='mean',
    margins=True
).round(0)
print(ct_val)

## 4. melt() — Wide to Long

In [ ]:
# Wide format: one column per quarter
wide_sales = pd.DataFrame({
    'salesperson': ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'Q1_revenue':  [120000, 90000, 150000, 80000, 110000],
    'Q2_revenue':  [130000, 95000, 145000, 85000, 120000],
    'Q3_revenue':  [140000, 100000, 160000, 88000, 125000],
    'Q4_revenue':  [160000, 110000, 175000, 92000, 135000],
})

print("Wide format:")
print(wide_sales)

In [ ]:
# melt() — unpivot wide → long
long_sales = wide_sales.melt(
    id_vars='salesperson',          # columns to keep
    value_vars=['Q1_revenue', 'Q2_revenue', 'Q3_revenue', 'Q4_revenue'],  # columns to melt
    var_name='quarter',             # name for the new "variable" column
    value_name='revenue'            # name for the values
)

# Clean up quarter names
long_sales['quarter'] = long_sales['quarter'].str.replace('_revenue', '')

print("Long format:")
print(long_sales.sort_values(['salesperson', 'quarter']))

In [ ]:
# When to use melt:
# - Feeding data to matplotlib / seaborn (needs long format)
# - Data arrives as columns but you need rows for groupby

# Aggregation on long format
print(long_sales.groupby('quarter')['revenue'].sum())

## 5. stack() and unstack() — MultiIndex Reshaping

In [ ]:
# stack(): move column level → row level (wide → long)
wide = summary.pivot(index='salesperson', columns='category', values='revenue').round(0)
print("Wide:")
print(wide)

long = wide.stack()  # category becomes a row level
long.name = 'revenue'
print("\nStacked (long):")
print(long.head(8))

In [ ]:
# unstack(): move row level → column level (long → wide)
wide_again = long.unstack('category')
print("Unstacked (back to wide):")
print(wide_again)

## 6. wide_to_long() — Structured Panel Data

In [ ]:
# wide_to_long: for structured column naming like 'metric_year'
panel = pd.DataFrame({
    'company':     ['Acme', 'Beta', 'Gamma'],
    'revenue_2021': [500, 300, 450],
    'revenue_2022': [550, 320, 480],
    'revenue_2023': [600, 350, 510],
    'profit_2021':  [50, 30, 45],
    'profit_2022':  [55, 32, 48],
    'profit_2023':  [60, 35, 51],
})

long_panel = pd.wide_to_long(
    panel,
    stubnames=['revenue', 'profit'],  # prefix
    i='company',                       # id var
    j='year',                          # the suffix becomes this column
    sep='_',
).reset_index()

print(long_panel.sort_values(['company', 'year']))

## 7. pivot_table vs groupby — Key Difference

In [ ]:
# groupby output: always long form — one column per aggregation
grp_result = (
    sales.groupby(['region', 'category'])['revenue']
    .sum()
    .reset_index()
)
print("groupby output (long):")
print(grp_result.head())

# pivot_table output: wide form — categories as columns
pt_result = pd.pivot_table(
    sales, values='revenue', index='region',
    columns='category', aggfunc='sum'
).round(0)
print("\npivot_table output (wide):")
print(pt_result)

## 8. Real-World — Sales Dashboard

In [ ]:
# Full dashboard: revenue share by region × category
revenue_matrix = pd.pivot_table(
    sales,
    values='revenue',
    index='region',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)

# Row % share (each cell / row total)
share_matrix = revenue_matrix.div(revenue_matrix['Total'], axis=0).drop(columns='Total').round(3) * 100

print("Revenue (₹):")
print(revenue_matrix.round(0))

print("\nCategory share within region (%):")
print(share_matrix)

In [ ]:
# YoY growth: pivot year as columns, compute % change
yoy = (
    sales.groupby(['year', 'category'])['revenue']
    .sum()
    .unstack('year')
    .round(0)
)
yoy.columns = [f'revenue_{y}' for y in yoy.columns]
yoy['yoy_growth_pct'] = ((yoy['revenue_2023'] - yoy['revenue_2022']) / yoy['revenue_2022'] * 100).round(1)
print(yoy)